# 01 — Exploratory Data Analysis: LaLiga Player Performance & Market Value

**Author:** Juan Sebastian Marcial
**Dataset:** LaLiga 2024-25 season (Transfermarkt)
**Objective:** Understand the structure of the data, identify patterns in player performance, and explore relationships between on-pitch metrics and market valuations.

---

### Table of Contents
1. [Setup & Data Loading](#1)
2. [Data Overview & Cleaning](#2)
3. [Descriptive Statistics](#3)
4. [Market Value Distribution](#4)
5. [Age & Position Analysis](#5)
6. [Performance Metrics vs Market Value](#6)
7. [Correlation Analysis](#7)
8. [Team-Level Insights](#8)
9. [Key Takeaways](#9)

<a id='1'></a>
## 1. Setup & Data Loading

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data_collection import load_dataset
from src.feature_engineering import engineer_features
from src.visualization import (
    set_football_style,
    plot_market_value_distribution,
    plot_age_distribution,
    plot_goals_vs_market_value,
    plot_correlation_heatmap,
    plot_value_by_position,
    plot_value_by_age_bucket,
    plot_team_squad_value,
    plot_performance_vs_value,
)

warnings.filterwarnings("ignore")
set_football_style()

%matplotlib inline

In [ ]:
# Load the real Transfermarkt dataset
df_raw = load_dataset()
print(f"Dataset loaded: {len(df_raw)} players from {df_raw['team'].nunique()} teams, {df_raw.shape[1]} columns")

Dataset loaded: 254 players from 21 teams, 15 columns


<a id='2'></a>
## 2. Data Overview & Cleaning

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   player_name         254 non-null   object
 1   team                254 non-null   object
 2   position            254 non-null   object
 3   sub_position        254 non-null   object
 4   age                 254 non-null   int64
 5   games_played        254 non-null   int64
 6   minutes_played      254 non-null   int64
 7   goals               254 non-null   int64
 8   assists             254 non-null   int64
 9   yellow_cards        254 non-null   int64
 10  red_cards           254 non-null   int64
 11  market_value_eur    254 non-null   int64
 12  height_cm           248 non-null   float64
 13  nationality         254 non-null   object
 14  preferred_foot      249 non-null   object
dtypes: float64(1), int64(7), object(7)


In [ ]:
# Top players by market value
df_raw[["player_name", "team", "position", "age", "games_played",
        "minutes_played", "goals", "assists", "market_value_eur"]].head()

         player_name             team  position  age  games_played  minutes_played  goals  assists  market_value_eur
0    Jude Bellingham      Real Madrid  Midfield   21            31            2493      9        9         160000000
1  Federico Valverde      Real Madrid  Midfield   26            36            3034      6        4         120000000
2     Julián Alvarez  Atletico Madrid    Attack   24            37            2519     17        4         100000000
3    Alejandro Balde     FC Barcelona  Defender   21            32            2293      0        4          60000000
4      Pablo Barrios  Atletico Madrid  Midfield   21            31            2337      1        4          60000000


In [ ]:
# Data quality checks
print("Missing values per column:")
print(df_raw.isnull().sum())
print(f"\nDuplicate player names: {df_raw['player_name'].duplicated().sum()}")
print(f"Players with 0 minutes: {(df_raw['minutes_played'] == 0).sum()}")
print(f"Teams represented: {df_raw['team'].nunique()}")

Missing values per column:
player_name         0
team                0
position            0
sub_position        0
age                 0
games_played        0
minutes_played      0
goals               0
assists             0
yellow_cards        0
red_cards           0
market_value_eur    0
height_cm           6
nationality         0
preferred_foot      5

Duplicate player names: 1
Players with 0 minutes: 0
Teams represented: 21


In [ ]:
# Apply feature engineering pipeline
df = engineer_features(df_raw, min_minutes=450)
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(f"After feature engineering: {len(df)} players, {df.shape[1]} columns")
print(f"New columns: {new_cols}")

After feature engineering: 254 players, 21 columns
New columns: ['goals_per90', 'assists_per90', 'goal_contribution_per90', 'age_bucket', 'position_group', 'log_market_value']


<a id='3'></a>
## 3. Descriptive Statistics

In [ ]:
df[["age", "minutes_played", "market_value_eur", "goals", "assists"]].describe().round(2)

          age  minutes_played  market_value_eur   goals  assists
count  254.00          254.00      2.540000e+02  254.00   254.00
mean    25.46         1312.59      8.430610e+06    1.58     1.12
std      4.59         1021.37      1.703891e+07    3.14     1.73
min     18.00            3.00      2.500000e+04    0.00     0.00
25%     22.00          386.00      1.000000e+06    0.00     0.00
50%     25.00         1127.00      2.500000e+06    0.00     0.00
75%     29.00         2172.00      8.750000e+06    2.00     2.00
max     39.00         3420.00      1.600000e+08   21.00     9.00


In [ ]:
print("Position distribution:")
print(df["position"].value_counts())
print(f"\nPosition group distribution:")
print(df["position_group"].value_counts())

Position distribution:
position
Defender      91
Attack        80
Midfield      63
Goalkeeper    20

Position group distribution:
position_group
Defender      91
Forward       80
Midfielder    63
Goalkeeper    20


<a id='4'></a>
## 4. Market Value Distribution

Market value is heavily right-skewed — a small number of star players command disproportionately high valuations, consistent with real Transfermarkt data.

In [ ]:
fig, ax = plot_market_value_distribution(df, log_scale=True)
plt.show()

<Figure>

In [ ]:
percentiles = [10, 25, 50, 75, 90, 95, 99]
print("Market Value Percentiles (EUR):")
for p in percentiles:
    val = df["market_value_eur"].quantile(p / 100)
    print(f"  {p}th percentile: €{val:>13,.0f}")
print(f"\nSkewness: {df['market_value_eur'].skew():.2f}")
print(f"Kurtosis: {df['market_value_eur'].kurtosis():.2f}")

Market Value Percentiles (EUR):
  10th percentile: €      500,000
  25th percentile: €    1,000,000
  50th percentile: €    2,500,000
  75th percentile: €    8,750,000
  90th percentile: €   20,000,000
  95th percentile: €   35,000,000
  99th percentile: €   78,800,000

Skewness: 5.12
Kurtosis: 34.69


<a id='5'></a>
## 5. Age & Position Analysis

In [ ]:
fig, ax = plot_age_distribution(df)
plt.show()

<Figure>

In [ ]:
fig, ax = plot_value_by_position(df)
plt.show()

<Figure>

In [ ]:
fig, ax = plot_value_by_age_bucket(df)
plt.show()

<Figure>

In [ ]:
print("Median market value by age bucket:")
for bucket in ["Young", "Rising", "Peak", "Experienced", "Twilight"]:
    val = df[df["age_bucket"] == bucket]["market_value_eur"].median()
    print(f"{bucket:<15} €{val:>11,.0f}")

print("\nMedian market value by position group:")
for group in ["Forward", "Midfielder", "Defender", "Goalkeeper"]:
    val = df[df["position_group"] == group]["market_value_eur"].median()
    print(f"{group:<15} €{val:>11,.0f}")

Median market value by age bucket:
Young           €  1,500,000
Rising          €  4,000,000
Peak            €  3,000,000
Experienced     €  1,800,000
Twilight        €    900,000

Median market value by position group:
Forward         €  3,000,000
Midfielder      €  2,000,000
Defender        €  3,000,000
Goalkeeper      €  2,750,000


<a id='6'></a>
## 6. Performance Metrics vs Market Value

In [ ]:
fig, ax = plot_goals_vs_market_value(df, per90=True)
plt.show()

<Figure>

In [ ]:
fig, ax = plot_performance_vs_value(df, x_col="assists_per90",
    title="Assists per 90 vs Market Value", xlabel="Assists per 90")
plt.show()

<Figure>

<a id='7'></a>
## 7. Correlation Analysis

In [ ]:
corr_cols = ["age", "minutes_played", "market_value_eur",
             "goals_per90", "assists_per90"]
corr_with_mv = (
    df_filtered[corr_cols].corr()["market_value_eur"]
    .drop("market_value_eur")
    .sort_values(key=abs, ascending=False)
)
print("Top correlations with market_value_eur:\n")
for feat, corr in corr_with_mv.items():
    print(f"  {feat:<30} {corr:>6.2f}")

Top correlations with market_value_eur:

  age                             -0.34
  assists_per90                    0.26
  goals_per90                      0.25
  minutes_played                   0.23


<a id='8'></a>
## 8. Team-Level Insights

In [ ]:
fig, ax = plot_team_squad_value(df)
plt.show()

<Figure>

In [ ]:
team_summary = (
    df.groupby("team").agg(
        squad_value=("market_value_eur", "sum"),
        mean_age=("age", "mean"),
        n_players=("player_name", "count"),
    ).sort_values("squad_value", ascending=False)
)
print(team_summary.head(10))

Team                   Squad Value     Mean Age   Players
───────────────────────────────────────────────────────
Atletico Madrid        €  408.2M       27.1      21
Real Madrid            €  360.0M       22.6      5
Real Sociedad          €  159.5M       23.5      17
Real Betis             €  153.6M       24.9      11
Valencia Club de Fútbol S. A. D. €  114.6M       23.8      13
Sevilla                €  107.3M       24.6      17
Athletic Bilbao        €  106.0M       26.5      4
FC Barcelona           €  100.0M       20.5      2
Girona Fútbol Club S. A. D. €   87.8M       23.5      14
Mallorca               €   81.7M       26.4      27


<a id='9'></a>
## 9. Key Takeaways

### Data Source
- **254 real players** across 21 LaLiga teams, sourced from **Transfermarkt**
- Market values are real and heavily right-skewed — log transformation essential for modeling

### Structural Patterns
- **Forwards** command the highest median market value
- **Age curve is asymmetric:** Values rise sharply (18→25) but decline gradually (28→37)
- The market pays a premium for young players with upside potential

### Performance → Value Relationships
- **Goals and assists per 90** correlate positively with market value
- **Non-linear effects are likely:** age interacts with position

### Implications for Modeling
- Use **log-transformed market value** as the target
- Include **position-aware features** or position interactions
- Try **tree-based models** (Random Forest) to capture non-linear effects

---

*Next: [02_predictive_model.ipynb](./02_predictive_model.ipynb) — Building and evaluating a market value prediction model*